In [0]:
"""
REAL DATA - STEP 1: CREATE USERS TABLE FROM EXISTING REALISTIC_TRANSACTIONS

Extract the 100 customers from realistic_transactions table
Create a users profile table with customer data + account info

Run this in Databricks notebook
"""

import pandas as pd
import numpy as np
from datetime import datetime

print("="*70)
print("CREATING USERS TABLE FROM REALISTIC_TRANSACTIONS")
print("="*70)

# ============================================
# STEP 1: Load realistic_transactions
# ============================================

print("\n[LOAD] Loading realistic_transactions data...")

try:
    # Get unique customers and their stats
    realistic_df = spark.sql("""
    SELECT
        customer_id,
        MAX(account_age_days) as account_age_days,
        MAX(country) as home_country,
        AVG(amount) as avg_daily_spend,
        COUNT(*) as transaction_count
    FROM banking_agent_db.realistic_transactions
    GROUP BY customer_id
    """).toPandas()

    print(f"  ✓ Found {len(realistic_df)} unique customers")
    print(f"  ✓ Extracted account_age_days, country, and calculated avg_daily_spend")

except Exception as e:
    print(f"  ⚠️ Error loading realistic_transactions: {e}")
    print(f"  Make sure banking_agent_db.realistic_transactions exists")
    raise

# ============================================
# STEP 2: Enrich with user data
# ============================================

print("\n[CREATE] Creating user profiles...")

# Create user IDs from customer IDs
realistic_df['user_id'] = 'user_' + (realistic_df.index + 1).astype(str).str.zfill(3)

# Generate realistic names and emails
np.random.seed(42)

first_names = ['Rohan', 'Keerthan', 'Priya', 'Amit', 'Sneha', 'Rajesh', 'Anushka',
               'Vikram', 'Deepika', 'Nikhil', 'Pooja', 'Arjun', 'Divya', 'Sanjay',
               'Koushik', 'Lokesh', 'Raju', 'Ramu', 'Hari', 'Kiran']

last_names = ['Kumar', 'Singh', 'Patel', 'Sharma', 'Gupta', 'Reddy', 'Rao',
              'Iyer', 'Nair', 'Verma', 'Joshi', 'Bhatt', 'Kapoor', 'Desai']

realistic_df['name'] = [f"{np.random.choice(first_names)} {np.random.choice(last_names)}"
                        for _ in range(len(realistic_df))]

realistic_df['email'] = realistic_df['user_id'] + '@example.com'

# Generate account balances based on account age and spending
# Older accounts likely have more balance
realistic_df['balance'] = realistic_df['account_age_days'] * 10 + realistic_df['avg_daily_spend'] * 30
realistic_df['balance'] = realistic_df['balance'].clip(lower=5000, upper=500000)  # ₹5k to ₹500k

# Account status: mostly active, some frozen
realistic_df['account_status'] = np.random.choice(
    ['active', 'frozen'],
    size=len(realistic_df),
    p=[0.95, 0.05]  # 95% active, 5% frozen
)

# KYC verification: mostly verified
realistic_df['kyc_verified'] = np.random.choice(
    [True, False],
    size=len(realistic_df),
    p=[0.90, 0.10]  # 90% verified, 10% not verified
)

# Created at: distributed over time
realistic_df['created_at'] = [
    datetime.now() - pd.Timedelta(days=int(age))
    for age in realistic_df['account_age_days']
]

# ============================================
# STEP 3: Select and order columns
# ============================================

users_df = realistic_df[[
    'user_id', 'customer_id', 'name', 'email', 'balance',
    'account_status', 'kyc_verified', 'account_age_days', 'home_country',
    'avg_daily_spend', 'created_at', 'transaction_count'
]]

print(f"  ✓ Created {len(users_df)} user profiles")

# ============================================
# STEP 4: Display sample data
# ============================================

print("\n[SAMPLE] First 10 users:")
print(users_df.head(10).to_string(index=False))

# ============================================
# STEP 5: Statistics
# ============================================

print("\n[STATS] User statistics:")
print(f"  Total users: {len(users_df)}")
print(f"  Active accounts: {len(users_df[users_df['account_status'] == 'active'])}")
print(f"  Frozen accounts: {len(users_df[users_df['account_status'] == 'frozen'])}")
print(f"  KYC verified: {len(users_df[users_df['kyc_verified'] == True])}")
print(f"\n  Balance distribution (₹):")
print(f"    Min: ₹{users_df['balance'].min():,.2f}")
print(f"    Max: ₹{users_df['balance'].max():,.2f}")
print(f"    Mean: ₹{users_df['balance'].mean():,.2f}")
print(f"    Total: ₹{users_df['balance'].sum():,.2f}")
print(f"\n  Account age distribution (days):")
print(f"    Min: {users_df['account_age_days'].min()} days")
print(f"    Max: {users_df['account_age_days'].max()} days")
print(f"    Mean: {users_df['account_age_days'].mean():.0f} days")

# ============================================
# STEP 6: Save to Databricks
# ============================================

print("\n[SAVE] Saving to Databricks...")

try:
    # Drop existing table to avoid schema conflicts
    spark.sql("DROP TABLE IF EXISTS banking_agent_db.users")
    print(f"  ✓ Dropped existing users table (if any)")
    
    users_spark = spark.createDataFrame(users_df)
    users_spark.write.mode("overwrite").saveAsTable("banking_agent_db.users")

    print(f"  ✓ Created table: banking_agent_db.users")

    # Verify
    count = spark.sql("SELECT COUNT(*) as count FROM banking_agent_db.users").collect()[0]['count']
    print(f"  ✓ Verified: {count} users in table")

except Exception as e:
    print(f"  ⚠️ Error saving to Databricks: {e}")

# ============================================
# STEP 7: Create mapping table (customer_id → user_id)
# ============================================

print("\n[MAPPING] Creating customer_id to user_id mapping...")

mapping_df = users_df[['customer_id', 'user_id']].copy()

try:
    # Drop existing mapping table to avoid schema conflicts
    spark.sql("DROP TABLE IF EXISTS banking_agent_db.customer_user_mapping")
    print(f"  ✓ Dropped existing mapping table (if any)")
    
    mapping_spark = spark.createDataFrame(mapping_df)
    mapping_spark.write.mode("overwrite").saveAsTable("banking_agent_db.customer_user_mapping")

    print(f"  ✓ Created mapping table: banking_agent_db.customer_user_mapping")

except Exception as e:
    print(f"  ⚠️ Error creating mapping: {e}")

# ============================================
# SUMMARY
# ============================================

print("\n" + "="*70)
print("✓ STEP 1 COMPLETE")
print("="*70)

print(f"""
Users Table Created:
✓ Table: banking_agent_db.users
✓ Rows: {len(users_df)}
✓ Columns: user_id, customer_id, name, email, balance, account_status, kyc_verified, account_age_days, home_country, avg_daily_spend, created_at

Mapping Table Created:
✓ Table: banking_agent_db.customer_user_mapping
✓ Enables linking realistic_transactions (customer_id) → users (user_id)

Next Step: REAL_DATA_STEP_2_LINK_TRANSACTIONS_TO_USERS.py
""")


In [0]:
"""
REAL DATA - STEP 2: LINK REALISTIC_TRANSACTIONS TO USERS

Add user_id to all realistic_transactions using customer_id mapping.
Keep train/test split separate for model training.

Run this in Databricks notebook
"""

import pandas as pd

print("="*70)
print("LINKING REALISTIC_TRANSACTIONS TO USERS")
print("="*70)

# ============================================
# STEP 1: Load realistic_transactions and mapping
# ============================================

print("\n[LOAD] Loading data...")

try:
    # Load all three existing tables
    realistic_df = spark.sql("SELECT * FROM banking_agent_db.realistic_transactions").toPandas()
    train_df = spark.sql("SELECT * FROM banking_agent_db.transactions_train").toPandas()
    test_df = spark.sql("SELECT * FROM banking_agent_db.transactions_test").toPandas()
    mapping_df = spark.sql("SELECT * FROM banking_agent_db.customer_user_mapping").toPandas()

    print(f"  ✓ realistic_transactions: {len(realistic_df)} rows")
    print(f"  ✓ transactions_train: {len(train_df)} rows")
    print(f"  ✓ transactions_test: {len(test_df)} rows")
    print(f"  ✓ customer_user_mapping: {len(mapping_df)} rows")

except Exception as e:
    print(f"  ⚠️ Error loading data: {e}")
    raise

# ============================================
# STEP 2: Add user_id to all transactions
# ============================================

print("\n[ADD] Adding user_id column...")

# Merge with mapping to add user_id
realistic_df = realistic_df.merge(mapping_df, on='customer_id', how='left')
train_df = train_df.merge(mapping_df, on='customer_id', how='left')
test_df = test_df.merge(mapping_df, on='customer_id', how='left')

print(f"  ✓ Added user_id to realistic_transactions")
print(f"  ✓ Added user_id to transactions_train")
print(f"  ✓ Added user_id to transactions_test")

# ============================================
# STEP 3: Verify data integrity
# ============================================

print("\n[VERIFY] Checking data integrity...")

# Check for nulls
null_count = realistic_df['user_id'].isnull().sum()
if null_count > 0:
    print(f"  ⚠️ Warning: {null_count} null user_ids in realistic_transactions")
else:
    print(f"  ✓ No null user_ids in realistic_transactions")

null_count = train_df['user_id'].isnull().sum()
if null_count > 0:
    print(f"  ⚠️ Warning: {null_count} null user_ids in transactions_train")
else:
    print(f"  ✓ No null user_ids in transactions_train")

null_count = test_df['user_id'].isnull().sum()
if null_count > 0:
    print(f"  ⚠️ Warning: {null_count} null user_ids in transactions_test")
else:
    print(f"  ✓ No null user_ids in transactions_test")

# ============================================
# STEP 4: Display sample data
# ============================================

print("\n[SAMPLE] Sample transactions with user_id:")
print(realistic_df[['transaction_id', 'customer_id', 'user_id', 'amount', 'merchant', 'is_fraudulent']].head(10).to_string(index=False))

# ============================================
# STEP 5: Distribution statistics
# ============================================

print("\n[STATS] Transaction distribution by user:")
user_dist = realistic_df.groupby('user_id').agg({
    'transaction_id': 'count',
    'amount': ['sum', 'mean'],
    'is_fraudulent': 'sum'
}).round(2)

user_dist.columns = ['transaction_count', 'total_amount', 'avg_amount', 'fraudulent_count']
user_dist = user_dist.sort_values('transaction_count', ascending=False)

print(f"\n  Top 10 users by transaction count:")
print(user_dist.head(10).to_string())

print(f"\n  Total transactions: {len(realistic_df)}")
print(f"  Total users: {realistic_df['user_id'].nunique()}")
print(f"  Total amount: ₹{realistic_df['amount'].sum():,.2f}")
print(f"  Fraudulent: {realistic_df['is_fraudulent'].sum()}")
print(f"  Fraud rate: {realistic_df['is_fraudulent'].mean()*100:.2f}%")

# ============================================
# STEP 6: Train/Test split statistics
# ============================================

print("\n[SPLIT] Train/Test split verification:")
print(f"  Train set:")
print(f"    - Rows: {len(train_df)}")
print(f"    - Users: {train_df['user_id'].nunique()}")
print(f"    - Fraudulent: {train_df['is_fraudulent'].sum()}")
print(f"    - Fraud rate: {train_df['is_fraudulent'].mean()*100:.2f}%")

print(f"  Test set:")
print(f"    - Rows: {len(test_df)}")
print(f"    - Users: {test_df['user_id'].nunique()}")
print(f"    - Fraudulent: {test_df['is_fraudulent'].sum()}")
print(f"    - Fraud rate: {test_df['is_fraudulent'].mean()*100:.2f}%")

# ============================================
# STEP 7: Save updated tables to Databricks
# ============================================

print("\n[SAVE] Saving updated tables...")

try:
    # Drop existing tables
    spark.sql("DROP TABLE IF EXISTS banking_agent_db.realistic_transactions")
    spark.sql("DROP TABLE IF EXISTS banking_agent_db.transactions_train")
    spark.sql("DROP TABLE IF EXISTS banking_agent_db.transactions_test")
    print(f"  ✓ Dropped existing tables")
    
    # Save realistic_transactions with user_id
    realistic_spark = spark.createDataFrame(realistic_df)
    realistic_spark.write.mode("overwrite").saveAsTable("banking_agent_db.realistic_transactions")
    print(f"  ✓ Updated banking_agent_db.realistic_transactions")

    # Save train set with user_id
    train_spark = spark.createDataFrame(train_df)
    train_spark.write.mode("overwrite").saveAsTable("banking_agent_db.transactions_train")
    print(f"  ✓ Updated banking_agent_db.transactions_train")

    # Save test set with user_id
    test_spark = spark.createDataFrame(test_df)
    test_spark.write.mode("overwrite").saveAsTable("banking_agent_db.transactions_test")
    print(f"  ✓ Updated banking_agent_db.transactions_test")

except Exception as e:
    print(f"  ⚠️ Error saving tables: {e}")

# ============================================
# STEP 8: Create user-specific views for queries
# ============================================

print("\n[VIEWS] Creating user-specific views...")

try:
    # View for daily transfer usage per user
    spark.sql("""
    CREATE OR REPLACE VIEW banking_agent_db.daily_transfer_usage AS
    SELECT
        user_id,
        DATE(timestamp) as transaction_date,
        COUNT(*) as transaction_count,
        SUM(amount) as daily_total,
        SUM(CASE WHEN is_fraudulent = 1 THEN 1 ELSE 0 END) as fraudulent_count
    FROM banking_agent_db.realistic_transactions
    WHERE is_fraudulent = 0
    GROUP BY user_id, DATE(timestamp)
    """)
    print(f"  ✓ Created view: banking_agent_db.daily_transfer_usage")

    # View for user transaction history
    spark.sql("""
    CREATE OR REPLACE VIEW banking_agent_db.user_transaction_history AS
    SELECT
        user_id,
        transaction_id,
        amount,
        merchant,
        hour,
        device,
        is_fraudulent,
        timestamp,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY timestamp DESC) as recency_rank
    FROM banking_agent_db.realistic_transactions
    """)
    print(f"  ✓ Created view: banking_agent_db.user_transaction_history")

except Exception as e:
    print(f"  ⚠️ Error creating views: {e}")

# ============================================
# SUMMARY
# ============================================

print("\n" + "="*70)
print("✓ STEP 2 COMPLETE")
print("="*70)

print(f"""
Updated Tables:
✓ banking_agent_db.realistic_transactions (with user_id)
✓ banking_agent_db.transactions_train (with user_id) - {len(train_df)} rows
✓ banking_agent_db.transactions_test (with user_id) - {len(test_df)} rows

Created Views:
✓ banking_agent_db.daily_transfer_usage
✓ banking_agent_db.user_transaction_history

Key Metrics:
✓ Total users: {realistic_df['user_id'].nunique()}
✓ Total transactions: {len(realistic_df)}
✓ Fraudulent: {realistic_df['is_fraudulent'].sum()} ({realistic_df['is_fraudulent'].mean()*100:.2f}%)
✓ Total amount: ₹{realistic_df['amount'].sum():,.2f}

Next Step: REAL_DATA_STEP_3_ADD_METHODS_TO_ORCHESTRATOR.py
  (Add load_user_data() and validate_transaction() methods)
""")